# 09 — Is difficulty a property of the series, or of the week?

Notebooks 05–08 took Chronos-2 as far as this problem seems to allow: context length swept and chosen per
horizon segment, the two winners stitched, a LoRA fine-tune on top. Covariates lost, cross-learning lost,
every encoding of the blank periods lost. One lever is left — **treat different series differently**:
fine-tune per group, cross-learn inside a group, one model per group.

That lever has a precondition, and it is testable before building anything. For a static grouping to help,
**difficulty has to be a stable property of a series**. If the items that are hard to forecast this week are
not the ones that were hard last week, then no grouping — hand-made, clustered or learned — has anything to
hold on to. That is a ceiling argument, and it costs a handful of extra forecast runs rather than a month of
clustering.

**Method.** Forecast from 24 origins, one week apart, and study the error of each (series, origin) pair.
Decompose it:

$$x_{f,o} = \mu + \alpha_f + \beta_o + \varepsilon_{f,o}$$

$\alpha_f$ is the series — by construction the only part a static grouping could exploit. $\beta_o$ is the
week: some weeks are hard for everyone. $\varepsilon_{f,o}$ is everything left, and it gets split further
into a part that survives to the next origin (persistent), a part that does not (transient), and the
measurement noise of the cell itself, read off the odd and even days of each window.

**Decision rule, fixed before looking.** Median Spearman between consecutive origins $\geq 0.6$ *and* at
least 35% of the variance on the series → difficulty is persistent, a grouping has something real. Spearman
$\leq 0.3$ *or* under 20% on the series → it is not, and no static grouping can help. Anything between →
partial, weigh it against the complexity. The rule lives in `persistence.verdict` so it cannot be read
wishfully after the fact.

**Three choices that make the measurement honest**, all argued in the module docstring: the **zero-shot**
model only (the stitched best model involves a checkpoint fine-tuned on history up to the test origin —
walking it backwards would let it see the future of every earlier window); origins counted back from the
**validation** origin, so the test window is never read; **non-overlapping** scored windows, so the lag-1
autocovariance is not an artefact of overlap.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from m5_chronos2.data import ARTIFACTS_DIR, roster_name
from m5_chronos2.evaluation import actuals, make_windows
from m5_chronos2.chronos2 import experiment_path, history, resolve_device
from m5_chronos2.persistence import (
    N_ORIGINS, SEGMENT_DAYS,
    bootstrap_shares, build_panel, cell_wql, decompose, origin_grid, rank_agreement,
    scored_cells, simulate_panel, spearman, transient_ratio, verdict,
)
from m5_chronos2.plots import (
    GRID_COLOR, MAIN_COLOR, MUTED_TEXT_COLOR, TEXT_COLOR, plot_fan, use_blog_style,
)

use_blog_style()

## Settings

In [ ]:
SAMPLE_SIZE = 500
SEED = 42
NAME = roster_name(SAMPLE_SIZE, SEED)

# The context length notebook 07 chose for days 1-7 — the segment this 7-day sweep scores.
choice = json.loads((ARTIFACTS_DIR / f"context_choice_{NAME}.json").read_text())
CONTEXT = int(choice["early_context"])

N_BOOTSTRAP = 1000     # resamples of the series, for the confidence intervals
WORST_DECILE = 0.10    # "the hardest tenth of series": does it survive to the next origin?

print(f"{N_ORIGINS} origins, {SEGMENT_DAYS} days apart, vanilla Chronos-2 at context {CONTEXT}")

In [ ]:
sales = pl.read_parquet(ARTIFACTS_DIR / f"sales_flags_{NAME}.parquet")
truth = actuals(sales)
VALIDATION_ORIGIN = make_windows().filter(pl.col("role") == "validation")["origin"][0]

ORIGINS = origin_grid(VALIDATION_ORIGIN, N_ORIGINS, SEGMENT_DAYS)
print(f"origins from {ORIGINS[0]} to {ORIGINS[-1]}")

# Every origin needs at least CONTEXT days of history behind it, or the earliest ones are
# quietly forecasting from less than the later ones and the panel is not comparable.
earliest_day = sales["date"].min()
room = (ORIGINS[0] - earliest_day).days
print(f"history available at the earliest origin: {room} days (context asks for {CONTEXT})")
assert room >= CONTEXT, "the earliest origin has less history than the context length"

## 0. Does the estimator work at all?

Before the real data, the positive control — the step most of these analyses skip, and the one that decides
whether a negative result means anything. `simulate_panel` builds a panel from known components at this
panel's size and noise level; the decomposition has to recover them.

This also exposes the one limit of the method worth knowing about: a difficulty that *drifts slowly* is
partly indistinguishable from a constant one over a panel of finite span, so it is shared between the series
term and the persistent term. Their **sum** is what is identified, and that is why `shares()` also reports
`stable`.

In [ ]:
control, planted = simulate_panel(n_series=250, n_origins=N_ORIGINS, seed=SEED)
control_result = decompose(control)
recovered = control_result.shares()

components = ["series", "origin", "persistent", "transient", "noise"]
control_table = pl.DataFrame({
    "component": components,
    "planted": [planted[c] for c in components],
    "recovered": [recovered["series_adjusted" if c == "series" else c] for c in components],
}).with_columns(
    (100 * pl.col("planted")).round(1).alias("planted %"),
    (100 * pl.col("recovered")).round(1).alias("recovered %"),
).drop("planted", "recovered")
print(f"raw (unadjusted) series share: {100 * recovered['series']:.1f}% — inflated by the noise in each alpha_f")
control_table

In [ ]:
# Where does a slow drift land? Sweep its autocorrelation, holding the planted amount fixed.
rows = []
for rho in [0.0, 0.3, 0.6, 0.85, 0.97]:
    simulated, truth_shares = simulate_panel(n_series=250, sd_series=0.0, sd_persistent=0.35,
                                             rho=rho, seed=SEED + 1)
    got = decompose(simulated).shares()
    rows.append({"rho": rho, "planted drift %": 100 * truth_shares["persistent"],
                 "-> series %": 100 * got["series_adjusted"], "-> persistent %": 100 * got["persistent"],
                 "sum %": 100 * got["stable"]})
pl.DataFrame(rows).with_columns(
    [pl.col(name).round(1) for name in ["planted drift %", "-> series %", "-> persistent %", "sum %"]]
)

The split moves with the timescale; the sum does not. Two readings follow, and both are used below: the
**series** share answers "could a grouping fitted once and left alone help", the **sum** answers "is there
anything stable at all, even if it has to be refitted".

Two more controls, run silently: a panel with no series effect, where the bias adjustment has to return 0;
and a white-noise panel, where the persistence correction has to return 0 rather than the negative value
double-centring induces.

In [ ]:
flat, _ = simulate_panel(n_series=250, sd_series=0.0, sd_persistent=0.0, seed=SEED + 2)
flat_shares = decompose(flat).shares()
print(f"no series effect, no drift:  raw series {100 * flat_shares['series']:.2f}%"
      f"  ->  adjusted {100 * flat_shares['series_adjusted']:.2f}%"
      f"  |  persistent {100 * flat_shares['persistent']:+.2f}%   (both truths: 0%)")

white, _ = simulate_panel(n_series=400, sd_series=0.0, sd_origin=0.0, sd_persistent=0.0, seed=SEED + 3)
white_result = decompose(white)
from m5_chronos2.persistence import _lag1_autocovariance
print(f"white noise only:  raw lag-1 autocovariance {_lag1_autocovariance(white_result.eps):+.5f}"
      f"  ->  corrected {white_result.v_persistent:+.5f}   (truth: 0)")

## 1. The panel

24 forecasts, read back from `artifacts/experiments/`. They are produced by the `origin_sweep` stage on the
tower (`uv run python scripts/run_stage.py --stages origin_sweep`); this notebook computes nothing.

In [ ]:
paths = [experiment_path(f"origin_sweep_ctx{CONTEXT}_origin{origin}", NAME) for origin in ORIGINS]
missing = [path.name for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} forecast(s) missing, starting with {missing[0]}.\n"
        "Run:  uv run python scripts/run_stage.py --stages origin_sweep   (on the tower), then rsync back."
    )

sweep = pl.concat([pl.read_parquet(path) for path in paths])
print(f"{sweep.height:,} forecast rows over {sweep['origin'].n_unique()} origins")

In [ ]:
# The scoring conditions of notebook 03, applied to a 7-day window instead of 28.
cells = scored_cells(sales, ORIGINS)
errors = cell_wql(sweep, truth, cells)
panel = build_panel(errors)

n_series, n_origins = panel.shape
print(f"scorable pairs: {cells.height:,}  ->  balanced panel: {n_series} series x {n_origins} origins")
print(f"  {panel.dropped['series_seen']} series appear at least once; "
      f"{panel.dropped['series_kept']} are scorable at every origin")
print(f"  cells with an undefined WQL (no sales in the window): {panel.dropped['cells_with_zero_wql']:,}")
assert n_series >= 80, "panel too thin to carry a conclusion — widen the roster before reading on"

The balanced panel is the honest object: double-centring a grid with holes is a different estimator, and a
series missing from half the origins would have its $\alpha_f$ measured on a different set of weeks than its
neighbours'. The count dropped by the balance requirement is printed above rather than buried, because a
filter that quietly removes most of the roster changes the question being answered.

Here is the panel itself — rows are series, sorted by their average difficulty, columns are origins in time
order. A world in which difficulty is a property of the series looks like clean horizontal bands.

In [ ]:
def plot_panel(panel, result):
    order = np.argsort(result.alpha)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    image = ax.imshow(panel.x[order], aspect="auto", cmap="RdYlBu_r", interpolation="nearest",
                      vmin=np.quantile(panel.x, 0.02), vmax=np.quantile(panel.x, 0.98))
    ax.set_xlabel("origin (one week apart, oldest left)")
    ax.set_ylabel("series, sorted by average difficulty")
    ax.set_xticks(range(0, panel.shape[1], 4))
    ax.set_xticklabels([str(panel.origins[i]) for i in range(0, panel.shape[1], 4)], rotation=45, ha="right")
    ax.set_yticks([])
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="log WQL", shrink=0.85)
    return fig

result = decompose(panel)
plot_panel(panel, result)
plt.show()

## 2. Where the error lives

The variance budget, with confidence intervals from resampling **series** (not cells: the question is
whether another sample of items would give the same answer, and cells within a series are not independent —
that dependence is the thing being measured).

In [ ]:
intervals = bootstrap_shares(panel, n_draws=N_BOOTSTRAP, seed=SEED)
shares = result.shares()

budget = pl.DataFrame([
    {"component": name,
     "share %": 100 * shares[key],
     "CI low %": 100 * intervals[key][0],
     "CI high %": 100 * intervals[key][1]}
    for name, key in [
        ("series (raw)", "series"),
        ("series (bias-adjusted)", "series_adjusted"),
        ("origin", "origin"),
        ("series x origin", "interaction"),
        ("  — persistent", "persistent"),
        ("  — transient", "transient"),
        ("  — sampling noise", "noise"),
        ("stable (series + persistent)", "stable"),
    ]
]).with_columns(
    [pl.col(name).round(1) for name in ["share %", "CI low %", "CI high %"]]
)
print(f"noise floor measured on {result.n_noise_cells:,} of {n_series * n_origins:,} cells "
      f"(both halves of the window need a sale)")
budget

In [ ]:
def plot_budget(shares, intervals):
    """The budget as it balances: the raw components sum to 1 exactly.

    The bias-adjusted series share is drawn as a mark *inside* the series block rather
    than substituted for it. Substituting would leave the five numbers short of 100%,
    and renormalising them would spread the adjustment over the noise term, which did
    not cause it.
    """
    parts = [("series", "series", MAIN_COLOR),
             ("origin", "origin", "#1baf7a"),
             ("persistent", "persistent", "#eb6834"),
             ("transient", "transient", "#b23c17"),
             ("noise", "noise", GRID_COLOR)]
    values = [shares[key] for _, key, _ in parts]
    assert abs(sum(values) - 1) < 1e-9, f"the budget does not balance: {sum(values):.4f}"

    fig, ax = plt.subplots(figsize=(11, 2.4))
    left, below = 0.0, []
    for (label, _, color), value in zip(parts, values):
        ax.barh(0, value, left=left, color=color, height=0.5)
        if value >= 0.10:
            ax.text(left + value / 2, 0, f"{label}\n{100 * value:.0f}%", ha="center", va="center",
                    color="white" if color != GRID_COLOR else TEXT_COLOR, fontsize=9)
        else:
            below.append((left + value / 2, label, value, color))
        left += value

    # Narrow blocks get their label under the bar, with a leader, so they never overlap.
    for position, (centre, label, value, color) in enumerate(below):
        offset = -0.42 - 0.20 * (position % 2)
        ax.plot([centre, centre], [-0.26, offset + 0.05], color=color, linewidth=0.8)
        ax.text(centre, offset, f"{label} {100 * value:.0f}%", ha="center", va="top",
                color=color, fontsize=8)

    # Where the series block ends once the noise in each alpha_f is taken out of it.
    adjusted = shares["series_adjusted"]
    ax.plot([adjusted, adjusted], [-0.25, 0.25], color="white", linestyle="--", linewidth=1.4)
    ax.annotate(f"bias-adjusted {100 * adjusted:.0f}%", xy=(adjusted, 0.25),
                xytext=(adjusted, 0.62), ha="center", va="bottom", fontsize=8, color=TEXT_COLOR,
                arrowprops=dict(arrowstyle="-", color=TEXT_COLOR, linewidth=0.8))

    low, high = intervals["stable"]
    ax.text(1.0, -1.05, f"stable (series + persistent) {100 * shares['stable']:.0f}%, "
                        f"95% CI [{100 * low:.0f}%, {100 * high:.0f}%]",
            ha="right", va="top", fontsize=8, color=MUTED_TEXT_COLOR)

    ax.set_xlim(0, 1)
    ax.set_ylim(-1.1, 0.95)
    ax.set_yticks([])
    ax.set_xlabel("share of the total squared log error")
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return fig

plot_budget(shares, intervals)
plt.show()

## 3. The same question, without a variance model

A decomposition is a model and can be wrong. Rank agreement is not: order the series by error at one origin,
order them again at the next, and ask whether the two orderings agree. If this disagrees with section 2,
something above is broken.

In [ ]:
ranks = rank_agreement(panel, decile=WORST_DECILE)
print(f"median Spearman between consecutive origins : {ranks['consecutive_median']:+.2f}")
print(f"Spearman, first origin vs last              : {ranks['first_vs_last']:+.2f}")
print(f"hardest {WORST_DECILE:.0%} still hardest next origin   : {100 * ranks['worst_decile_survival']:.0f}%"
      f"   (a coin flip would give {100 * ranks['worst_decile_random']:.0f}%)")

In [ ]:
def plot_ranks(panel, ranks):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

    ax = axes[0]
    values = ranks["consecutive"]
    ax.plot(range(1, len(values) + 1), values, color=MAIN_COLOR, marker="o", markersize=4)
    ax.axhline(ranks["consecutive_median"], color=TEXT_COLOR, linestyle="--", linewidth=1)
    ax.axhspan(-0.5, 0.5, color=GRID_COLOR, alpha=0.6, zorder=0)
    ax.text(0.98, 0.04, "|rho| < 0.5: 'weak' for test-retest", transform=ax.transAxes,
            ha="right", va="bottom", color=MUTED_TEXT_COLOR, fontsize=8)
    ax.set_ylim(-1, 1)
    ax.set_xlabel("pair of consecutive origins")
    ax.set_ylabel("Spearman of the difficulty ranking")

    ax = axes[1]
    first, last = panel.x[:, 0], panel.x[:, -1]
    ax.scatter(first, last, s=10, color=MAIN_COLOR, alpha=0.6)
    limits = [min(first.min(), last.min()), max(first.max(), last.max())]
    ax.plot(limits, limits, color=MUTED_TEXT_COLOR, linewidth=1)
    ax.set_xlabel(f"log WQL at the first origin ({panel.origins[0]})")
    ax.set_ylabel(f"... at the last ({panel.origins[-1]})")
    ax.text(0.03, 0.95, f"Spearman {ranks['first_vs_last']:+.2f}", transform=ax.transAxes,
            va="top", color=TEXT_COLOR)
    fig.tight_layout()
    return fig

plot_ranks(panel, ranks)
plt.show()

## 4. Validating the residual of the residual

`transient` is a difference of differences: it is whatever is left after subtracting noise and persistence
from the interaction, so *any* leak — a broken origin, a series going dark, a join artefact — lands in it.
Before believing a large transient share, look at the cells it is built from.

For each cell, predict its residual from the neighbouring origins using the autocovariances already
estimated (nothing new is fitted) and square what is left, in units of the noise floor. A large ratio should
be one origin jumping off an otherwise flat trace, **with a visible event underneath**. If instead the
extreme cells are near-zero-volume series, the transient share is measuring data holes and the ceiling
argument has to be re-run on a cleaner panel.

In [ ]:
ratio = transient_ratio(panel, result)
worst = np.unravel_index(np.argmax(ratio), ratio.shape)
calm = np.unravel_index(np.argmin(ratio), ratio.shape)
print(f"largest t_ratio {ratio[worst]:,.0f}  — {panel.ids[worst[0]]} at {panel.origins[worst[1]]}")
print(f"smallest        {ratio[calm]:.2f}  — {panel.ids[calm[0]]} at {panel.origins[calm[1]]}")
print(f"median {np.median(ratio):.2f}; share of cells above 10x the noise floor: {100 * (ratio > 10).mean():.1f}%")

In [ ]:
def plot_cell(axes, panel, result, cell, color, label):
    series_index, origin_index = cell
    series_id, origin = panel.ids[series_index], panel.origins[origin_index]

    # Top: this series' error across every origin, against what the model of section 2 expects of it.
    ax = axes[0]
    expected = result.mu + result.alpha[series_index] + result.beta
    ax.plot(range(panel.shape[1]), panel.x[series_index], color=color, marker="o", markersize=3,
            label="observed log WQL")
    ax.plot(range(panel.shape[1]), expected, color=MUTED_TEXT_COLOR, linestyle="--", linewidth=1,
            label="mu + alpha_f + beta_o")
    ax.axvline(origin_index, color=color, linewidth=1, alpha=0.4)
    ax.set_title(f"{label}: {series_id}, origin {origin}", loc="left")
    ax.set_xlabel("origin")
    ax.set_ylabel("log WQL")
    ax.legend(fontsize=8, loc="best")

    # Bottom: what actually happened that week.
    past = (history(sales, origin, [series_id])
            .select("date", pl.col("sales").alias("y")).tail(8 * SEGMENT_DAYS))
    window = sweep.filter((pl.col("id") == series_id) & (pl.col("origin") == origin)).sort("date")
    window = window.filter((pl.col("date") - pl.col("origin")).dt.total_days() <= SEGMENT_DAYS)
    actual = truth.filter((pl.col("id") == series_id)
                          & pl.col("date").is_in(window["date"])).sort("date")
    plot_fan(axes[1], past, window, actual, color=color, label="forecast")
    axes[1].set_xlabel("date")

fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))
plot_cell(axes[:, 0], panel, result, worst, "#b23c17", "largest transient")
plot_cell(axes[:, 1], panel, result, calm, MAIN_COLOR, "smallest transient")
fig.tight_layout()
plt.show()

## 5. Verdict

The rule was fixed before the data was looked at; here it is applied.

In [ ]:
print(verdict(result, ranks, intervals))

In [ ]:
record = {
    "origins": [str(origin) for origin in ORIGINS],
    "n_origins": n_origins,
    "n_series": n_series,
    "segment_days": SEGMENT_DAYS,
    "context_length": CONTEXT,
    "model": "chronos2 vanilla (zero-shot)",
    "selected_on": f"origins counted back from the validation origin {VALIDATION_ORIGIN}",
    "shares": {key: round(value, 4) for key, value in shares.items()},
    "shares_ci": {key: [round(low, 4), round(high, 4)] for key, (low, high) in intervals.items()},
    "spearman_consecutive_median": round(ranks["consecutive_median"], 4),
    "spearman_first_vs_last": round(ranks["first_vs_last"], 4),
    "worst_decile_survival": round(ranks["worst_decile_survival"], 4),
    "worst_decile_random": round(ranks["worst_decile_random"], 4),
    "verdict": verdict(result, ranks, intervals).splitlines()[0],
}
path = ARTIFACTS_DIR / f"error_persistence_{NAME}.json"
path.write_text(json.dumps(record, indent=2))
print(f"Saved to {path.name}")
record["verdict"]

**How much to trust this.** Three limits, in decreasing order of how much they could change the answer.

1. **One model, one horizon segment.** The sweep uses the zero-shot model on days 1–7. A different model, or
   a longer horizon, could in principle have a different difficulty structure — though it would be odd for a
   fine-tune to create stability that the base model does not see.
2. **24 origins span about six months.** A difficulty that drifts on a yearly scale would read as a constant
   series effect here, which is precisely what the $\rho = 0.97$ row of section 0 shows. That bias is in the
   *favourable* direction: it inflates the series share, so a low series share is not an artefact of the
   span.
3. **The panel is balanced**, so it over-represents items that sell steadily enough to be scorable at all 24
   origins. Those are, if anything, the easiest items to group — another bias in the favourable direction.

Both biases push toward finding structure. Whatever the verdict above says, it says it against the odds.

## Notes

*Your observations here.*